# FastAPI Fundamentals

---

In this notebook, we will learn the fundamentals of **FastAPI**, the modern Python web framework we'll use to serve machine learning models as APIs.

We will cover:
- Installing FastAPI and Uvicorn
- Creating a minimal FastAPI application
- Defining `GET` and `POST` endpoints
- Understanding path and query parameters
- Working with request bodies and JSON
- Exploring the auto-generated Swagger documentation

> ⚠️ **Note:** FastAPI applications are Python scripts (`.py` files), not Jupyter Notebooks. You cannot run a web server from within a notebook in the traditional sense. This notebook explains the concepts and shows the code. The actual runnable application lives in the `app/` subfolder.

---

## 1. Installation
FastAPI requires two packages:
- `fastapi`: The framework itself
- `uvicorn`: An ASGI server that actually runs your FastAPI application.

```bash
uv add fastapi uvicorn
```

Think of it this way: FastAPI is the *recipe*, and Uvicorn is the *oven*. You write the API with FastAPI, and Uvicorn serves it to the world.

---

## 2. Your First API
Here is the simplest possible FastAPI application:

In [ ]:
# app/main.py
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def root():
    return {"message": "Hello, World!"}

Let's break this down:

| Line | What It Does |
| :--- | :--- |
| `from fastapi import FastAPI` | Imports the framework. |
| `app = FastAPI()` | Creates an application instance. This is the central object that holds all your routes. |
| `@app.get("/")` | A **decorator** that registers the function below as a handler for `GET` requests to the `/` path (the root URL). |
| `def root():` | A plain Python function. FastAPI calls it when someone visits `/`. |
| `return {"message": ...}` | FastAPI automatically converts Python dicts to **JSON** responses. |

### 2.1. Running the Application

To start the server, run this command from the terminal:

In [ ]:
uvicorn app.main:app --reload

| Part | Meaning |
| :--- | :--- |
| `app.main` | The Python module path: file `main.py` inside the `app/` folder. |
| `:app` | The FastAPI instance variable inside that file. |
| `--reload` | Auto-restarts the server when you edit the code (development only). |

After running this, you'll see:

In [ ]:
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)

Open `http://127.0.0.1:8000` in your browser and you'll see:

```json
{"message": "Hello, World!"}
```

---

## 3. GET Endpoints

`GET` requests are used to retrieve information. They don't send data in the request body.

### 3.1. Path Parameters

Path parameters are variables embedded in the URL path itself:

In [ ]:
@app.get("/items/{item_id}")
def read_item(item_id: int):
    return {"item_id": item_id}

- Visiting `/search?query=iris&limit=5` returns `{"query": "iris", "limit": 5}`
- `limit` has a default value of `10`, making it optional
- `query` has no default, making it **required**.

---

## 4. POST Endpoints and Request Bodies

`POST` requests are used to **send data** to the server. This is the method we'll use for model predictions. The client sends feature values, and the server returns a prediction.

FastAPI uses **Pydantic models** to define the shape of the request body:

In [ ]:
from pydantic import BaseModel

class Item(BaseModel):
    name: str
    price: float
    quantity: int = 1    # default value → optional field

@app.post("/items")
def create_item(item: Item):
    total = item.price * item.quantity
    return {"name": item.name, "total": total}

When a client sends a `POST` request to `/items` with this JSON body:

```json
{"name": "Widget", "price": 9.99, "quantity": 3}
```

FastAPI automatically:
1. **Parses** the JSON into an `ITEM` object.
2. **Validates** that `name` is a string, `price` is a float, `quantity` is an int.
3. **Returns a clear error** if the data is invalid (e.g., if `price` is `"hello"`)

### 4.1. What is Pydantic

**Pydantic** is a data validation library that FastAPI is built on. You define a Python class with  type hints, and Pydantic enforces them at runtime.

| Feature | What It Does |
| :--- | :--- |
| `name: str` | Field is required, must be a string |
| `price: float` | Field is required, must be a number |
| `quantity: int = 1` | Field is optional, defaults to 1 if not provided |

This is extremely powerful for ML APIs: you can define exactly what features your model expects, and FastAPI will reject any malformed input before it ever reaches your model.

---

## 5. Response Models

Just as you define the input shape, you can define the output shape using a `response_model`:

In [ ]:
class ItemResponse(BaseModel):
    name: str
    total: float
    
@app.post("/items", response_model=ItemResponse)
def create_item(item: Item):
    total = item.price * item.quantity
    return {"name": item.name, "total": total}

This ensures your API always returns a consistent, documented response structure. It also appears in the Swagger documentation so clients know exactly what to expect.

---

## 6. Auto-Generated Documentation

One of FastAPI's best features is that it automatically generates interactive API documentation from your code. No extra work needed. When your server is running, visit:

| URL | What You Get |
| :--- | :--- |
| `http://127.0.0.1:8000/docs` | **Swagger UI** — An interactive page where you can test your endpoints directly in the browser. You can fill in parameters, click "Execute", and see the response. |
| `http://127.0.0.1:8000/redoc` | **ReDoc** — A cleaner, read-only documentation page. Great for sharing with clients. |

Both are generated automatically from your Pydantic models and function signatures. The type hints you write aren't just for your IDE, they become the API's documentation.

---

## 7. Status Codes and Error Handling

HTTP responses include a **status code** that tells the client what happened:

| Code | Meaning | When To Use |
| :--- | :--- | :--- |
| **200** | OK | Request succeeded (default for FastAPI) |
| **201** | Created | A resource was created |
| **400** | Bad Request | Client sent invalid data |
| **404** | Not Found | The requested endpoint doesn't exist |
| **422** | Unprocessable Entity | Pydantic validation failed (FastAPI returns this automatically) |
| **500** | Internal Server Error | Something crashed on the server |

FastAPI handles most of this automatically. When Pydantic validation fails, the client gets a `422` response with a detailed message explaining which field is invalid and why.

You can also raise errors manually:

In [ ]:
from fastapi import HTTPException

@app.get("/items/{item_id}")
def read_item(item_id: int):
    if item_id not in database:
        raise HTTPException(status_code=404, detail="Item not found")
    return database[item_id]


---

## 8. Summary

| Concept | Key Takeaway |
| :--- | :--- |
| **FastAPI** | A modern, fast Python web framework built on Pydantic and type hints. |
| **Uvicorn** | The ASGI server that runs FastAPI apps. Start with `uvicorn app.main:app --reload`. |
| **`@app.get()`** | Defines a GET endpoint (retrieve data, no request body). |
| **`@app.post()`** | Defines a POST endpoint (send data, get a result). Used for predictions. |
| **Pydantic `BaseModel`** | Defines the shape and types of request/response data. Validation is automatic. |
| **`/docs`** | Auto-generated Swagger UI for interactive testing. |
| **`HTTPException`** | Raise custom error responses with specific status codes. |

---

**Next:** [Serving a Model with FastAPI](./02_serving_a_model_with_fastapi.ipynb) — Loading our saved Iris Pipeline and wrapping it in a prediction endpoint.